## Advanced: structure

https://docs.typesafe.ai/primitives/advanced

Az `instructions` és minden `criteria` elem lehet string, objektum, tömb vagy `null`. A kulcsneveket mi választjuk, a modell a neveket is látja.

In [ ]:
from dotenv import load_dotenv
from utils import call_jev

load_dotenv("../.env")

### Strukturált instructions

A `field` objektum leírja a vizsgált mezőt, a kérdés a kulcsaira hivatkozik (`extracted_value`, `field`), nem csak a state mezőire.

In [ ]:
payload = {
    "model": "jev-latest",
    "state": {
        "source_text": "Invoice #4471 issued March 3, 2026 to Beaver Dam Logistics for $12,840.00, net 30.",
    },
    "questions": {
        "invoice_number_is_correct": {
            "type": "noul",
            "instructions": {
                "field": {
                    "name": "invoice_number",
                    "type": "string",
                    "description": "The identifier printed on the invoice.",
                },
                "extracted_value": "4471",
                "question": "Does `extracted_value` match the `field` as it appears in `source_text`?",
            },
        },
        "customer_name": {
            "type": "choice",
            "instructions": {
                "field": {
                    "name": "customer_name",
                    "type": "string",
                    "description": "The organization the invoice was issued to.",
                },
                "question": "Which option is the value of `field` in `source_text`?",
            },
            "criteria": {
                "Beaver Logistics": None,
                "Dam Logistics": None,
                "Beaver Dam Logistics": None,
                "Beaver": None,
                "Dam": None,
            },
        },
        "amount_due": {
            "type": "score",
            "instructions": {
                "field": {
                    "name": "amount_due",
                    "type": "number",
                    "unit": "USD",
                    "description": "The total the invoice asks to be paid.",
                },
                "question": "How large is the `field` value in `source_text`?",
            },
            "criteria": [
                "Under $1,000",
                "$1,000 to $10,000",
                "$10,000 to $100,000",
                "$100,000 to $1,000,000",
                "Over $1,000,000",
            ],
        },
    },
}

result = call_jev(payload)
result

### Strukturált Choice opciók

Opciónként `what` / `not_for` / `examples`: mit fed le az opció, és mit **nem**. Mellette ugyanaz a kérdés sima opciókkal, összehasonlításnak.

In [ ]:
state = "I ordered the standing desk two weeks ago and tracking still says label created. Was I even charged?"

payload = {
    "model": "jev-latest",
    "state": state,
    "questions": {
        "department_plain": {
            "type": "choice",
            "instructions": "Which team should handle this message?",
            "criteria": {"billing": None, "orders": None, "account": None},
        },
        "department_rubric": {
            "type": "choice",
            "instructions": {
                "question": "Which team should handle this message?",
                "focus": "Classify the customer's primary request, not every topic mentioned.",
            },
            "criteria": {
                "billing": {
                    "what": "Charges, invoices, refunds, or subscriptions",
                    "not_for": "Order tracking or account access",
                    "examples": ["I was charged twice", "Where is my refund?"],
                },
                "orders": {
                    "what": "Order status, delivery, cancellation, or returns",
                    "not_for": "Charges or account access",
                    "examples": ["Where is my package?", "Cancel my order"],
                },
                "account": {
                    "what": "Login, password, profile, or security",
                    "not_for": "Charges or delivery",
                    "examples": ["I can't log in", "Change my email"],
                },
            },
        },
    },
}

result = call_jev(payload)
result

### Taxonómia bejárása

Az opció értéke a teljes részfa, így a modell látja, mi van egy ág alatt.

In [ ]:
payload = {
    "model": "jev-latest",
    "state": "32oz plastic bottle with a flip straw lid. Fits most bike cages.",
    "questions": {
        "department": {
            "type": "choice",
            "instructions": "Which top-level department does this product belong to?",
            "criteria": {
                "Sporting Goods": {
                    "Cycling": ["Bike Bottles & Cages", "Bike Lights", "Helmets"],
                    "Fitness": ["Yoga Mats", "Resistance Bands"],
                    "Outdoor": ["Tents", "Sleeping Bags", "Hydration Packs"],
                },
                "Home & Kitchen": {
                    "Drinkware": ["Water Bottles", "Travel Mugs", "Tumblers"],
                    "Cookware": ["Pots & Pans", "Bakeware"],
                },
                "Baby & Toddler": ["Sippy Cups", "Bottle Warmers", "Bibs"],
            },
        },
    },
}

result = call_jev(payload)
result

### Strukturált Score szintek

In [ ]:
payload = {
    "model": "jev-latest",
    "state": "Fixed the null check in the payment handler. Also refactored the retry loop while I was in there, and bumped the SDK version since the old one had that timeout bug.",
    "questions": {
        "pr_scope": {
            "type": "score",
            "instructions": {
                "question": "How focused is this pull request description on a single change?",
                "note": "Judge the number of independent changes, not the size of any one change.",
            },
            "criteria": [
                {
                    "summary": "One change, clearly stated",
                    "signals": ["A single fix or feature", 'Nothing described as "also" or "while I was in there"'],
                },
                {
                    "summary": "One main change plus a small related tweak",
                    "signals": ["A primary change and one minor adjacent edit", "The tweak supports the main change"],
                },
                {
                    "summary": "Several independent changes bundled together",
                    "signals": ["Two or more unrelated fixes or features", "Changes that could each be their own PR"],
                },
            ],
        },
    },
}

result = call_jev(payload)
result

### Strukturált Noul criteria

In [ ]:
payload = {
    "model": "jev-latest",
    "state": {
        "sender": {"display_name": "Beaver Dam Builders Ltd.", "email": "donotreply@payroll.example"},
        "message": "Your Q3 bonus is ready. Reply with your login password so we can verify your identity and release the funds.",
    },
    "questions": {
        "requests_credentials": {
            "type": "noul",
            "instructions": {
                "question": "Does the `message` ask the recipient to disclose a sensitive credential?",
                "inspect": "message",
                "focus": "Look for a request to send the credential itself, not a request to change or reset it.",
            },
            "criteria": {
                "true": {
                    "what": "Asks the recipient to reply with, type, or send a password, PIN, one-time code, or other security sensitive answer",
                    "examples": ["Reply with your password", "Send us the 6-digit code you just received"],
                },
                "false": {
                    "what": "No sensitive credential is requested",
                    "examples": ["Reset your password from the settings page", "Your statement is ready"],
                },
            },
        },
    },
}

result = call_jev(payload)
result